# Unit 9 solutions: build a line, follow it, and measure it honestly

These reference solutions build the lesson's line solver from its pieces, check each piece where the answer is known (the spline's weights, a circle, the QR identity), and then race the line, always against the race's own line.
Each exercise's statement is repeated, followed by a prediction made before running anything, then the solution.
Every race is seeded, so every number below is reproducible, and executable asserts check each answer against an independent derivation.
Every number is a number about racekit's **carla-like car**: a model, not the truth.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_08, UNIT_09 = AGENTS / "unit-08.py", AGENTS / "unit-09.py"
workdir = Path(mkdtemp(prefix="roar-unit-09-sol-"))
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the cross-track error from the race's own line
REACHED_RADIUS = 3.0  # m: the agents' "waypoint reached" radius
TICK = 0.05 + 1e-9  # race times are pinned to one control tick

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
# (monza-like's section 0 never slid in its probe, so its number is a lower bound; the planner uses it as measured.)
MEASURED_MU_G = {
    "test-circuit": (7.89, 9.72, 8.65, 7.87, 8.30, 8.14),
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "monza-like": (5.38, 8.96, 9.12, 9.14, 9.70, 7.97),
}


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def menger(x, y):
    # Unit 2's signed Menger curvature at every point of a closed polyline (NumPy arrays; left turns positive).
    xa, ya, xc, yc = np.roll(x, 1), np.roll(y, 1), np.roll(x, -1), np.roll(y, -1)
    twice_area = (x - xa) * (yc - ya) - (y - ya) * (xc - xa)
    sides = np.hypot(x - xa, y - ya) * np.hypot(xc - x, yc - y) * np.hypot(xc - xa, yc - ya)
    return 2.0 * twice_area / sides


def line_summary(x, y):
    # (total squared curvature sum kappa^2 ds, lap length in m, tightest radius in m) of a closed line.
    kappa = menger(x, y)
    ds = np.hypot(np.roll(x, -1) - x, np.roll(y, -1) - y)
    return float(np.sum(kappa ** 2 * ds)), float(ds.sum()), float(1.0 / np.max(np.abs(kappa)))

## Exercise 1 - The B-spline's four weights

Waypoint i of an n-waypoint lap with m = n // step control values sits at t = i·m / n; with j = ⌊t⌋ and u = t − j, its offset is w_0 c_{j−1} + w_1 c_j + w_2 c_{j+1} + w_3 c_{j+2} (indices mod m), where

w_0 = (1 − u)³ / 6, w_1 = (3u³ − 6u² + 4) / 6, w_2 = (−3u³ + 3u² + 3u + 1) / 6, w_3 = u³ / 6.

Write:

- `spline_weights(u)`: returns the tuple (w_0, w_1, w_2, w_3);
- `spline_basis(n, step=10)`: returns an n × m NumPy array whose entry [i, k] is the weight of control value k in waypoint i's offset (add each of the four weights into column (j − 1 + k) mod m, k = 0 … 3).

Define:

- `spline_weights`, `spline_basis`: the functions;
- `weights_at`: a dict from u to `spline_weights(u)` for u = 0.0, 0.25, 0.5 and 0.75;
- `smallest_weight`: the smallest of the four weights over the 101 values `np.linspace(0.0, 1.0, 101)`;
- `worst_sum_error`: the largest |w_0 + w_1 + w_2 + w_3 − 1| over the same 101 values;
- `toy_basis`: `spline_basis(40, 10)` (so m = 4);
- `nonzeros_per_row`: the sorted list of the distinct numbers of entries > 0 in a row of `toy_basis`;
- `toy_control`: the NumPy array (2.0, −1.0, 3.0, 0.5), and `toy_offsets = toy_basis @ toy_control`;
- `bound_holds`: `True` when every |toy_offsets[i]| is at most the largest |toy_control[k]| over the columns k where row i of `toy_basis` is > 0;
- `why_bounds_carry`: one or two sentences: why does a limit |c_k| ≤ h on every control value guarantee |n_i| ≤ h at every waypoint, and which two properties of the weights does that need?

**Prediction.** At u = 0 the weights are 1/6, 4/6, 1/6 and 0, so a waypoint that sits exactly on a control value averages only three of them; at u = 0.5 they are 1/48, 23/48, 23/48, 1/48 (symmetric), and u = 0.75 is u = 0.25 reversed.
The smallest weight over [0, 1] is exactly 0 (w_3 at u = 0, w_0 at u = 1), and the four always sum to 1, so the sum error is rounding only.
With n = 40 and m = 4 a control value sits every 10 waypoints, so rows 0, 10, 20 and 30 have three entries above 0 and all the others four: `[3, 4]`.

In [ ]:
def spline_weights(u):
    return ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
            u ** 3 / 6)


def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        weights = spline_weights(t - j)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


weights_at = {u: spline_weights(u) for u in (0.0, 0.25, 0.5, 0.75)}
grid = np.linspace(0.0, 1.0, 101)
smallest_weight = min(min(spline_weights(u)) for u in grid)
worst_sum_error = max(abs(sum(spline_weights(u)) - 1.0) for u in grid)
toy_basis = spline_basis(40, 10)
nonzeros_per_row = []
for row in toy_basis:
    count = int(np.sum(row > 0))
    if count not in nonzeros_per_row:
        nonzeros_per_row.append(count)
nonzeros_per_row = sorted(nonzeros_per_row)
toy_control = np.array((2.0, -1.0, 3.0, 0.5))
toy_offsets = toy_basis @ toy_control
bound_holds = False not in [abs(toy_offsets[i]) <= np.max(np.abs(toy_control[toy_basis[i] > 0]))
                            for i in range(len(toy_offsets))]
why_bounds_carry = (
    "Every offset is a weighted average of the control values in its row: the weights are never negative "
    "and they sum to 1, so |n_i| <= w_0|c_{j-1}| + ... + w_3|c_{j+2}| <= (w_0 + ... + w_3) h = h, and a "
    "limit on every control value is a limit on every waypoint."
)

for u, weights in weights_at.items():
    print(f"u = {u:4.2f}: weights {tuple(round(w, 6) for w in weights)}, sum {sum(weights):.15f}")
print(f"smallest weight over 101 u: {smallest_weight}; worst |sum - 1|: {worst_sum_error:.1e}")
print(f"toy basis {toy_basis.shape}; nonzeros per row: {nonzeros_per_row}")
print("toy offsets:", np.round(toy_offsets, 4))
print(f"every offset inside its own control values' largest size: {bound_holds}")
print(why_bounds_carry)

# independent checks: the hand values of the weights, and the toy basis's structure
assert weights_at[0.0] == (1 / 6, 4 / 6, 1 / 6, 0.0)
assert max(abs(a - b) for a, b in zip(weights_at[0.5], (1 / 48, 23 / 48, 23 / 48, 1 / 48))) < 1e-15
assert max(abs(a - b) for a, b in zip(weights_at[0.25], (27 / 384, 235 / 384, 121 / 384, 1 / 384))) < 1e-15
assert smallest_weight == 0.0 and worst_sum_error < 1e-15
assert toy_basis.shape == (40, 4) and np.min(toy_basis) >= 0.0
assert np.max(np.abs(toy_basis.sum(axis=1) - 1.0)) < 1e-15
assert nonzeros_per_row == [3, 4]
# waypoint 0 sits on control value 0 (u = 0): 1/6, 4/6, 1/6 of c_3, c_0, c_1
assert abs(toy_offsets[0] - (0.5 / 6 + 4 * 2.0 / 6 - 1.0 / 6)) < 1e-15
assert bound_holds and np.max(np.abs(toy_offsets)) < np.max(np.abs(toy_control))

## Exercise 2 - Normals and second differences on a circle

On a circle the answers are known, so it is a good place to check the tools.
Make a circle of radius 50 m around the origin, 300 points anticlockwise: `theta = 2 * np.pi * np.arange(300) / 300`, `cx = 50 * np.cos(theta)`, `cy = 50 * np.sin(theta)`.

Write:

- `normals(x, y)`: the left unit normal at every point of a closed polyline, as two NumPy arrays (nx, ny): the chord from point i − 1 to point i + 1 is (d_x, d_y) = (x_{i+1} − x_{i−1}, y_{i+1} − y_{i−1}) with the ends wrapping round, h = √(d_x² + d_y²), and the normal is (−d_y / h, d_x / h);
- `second_difference(values)`: v_{i+1} − 2 v_i + v_{i−1} for every i, the ends wrapping round, along the **first axis** (so a 1-D array and every column of a 2-D array both work; `np.roll(values, -1, axis=0)` is v_{i+1}).

Define:

- `normals`, `second_difference`: the functions;
- `cnx`, `cny`: the circle's normals;
- `points_inward`: `True` when every normal points at the centre, that is, (cnx, cny) is within 1e-9 of (−cx / 50, −cy / 50) at every point;
- `inner_x`, `inner_y`: the circle's points moved n = 3.0 m along their normals (the same offset everywhere);
- `inner_radius`: the mean distance of the moved points from the origin;
- `inner_curvature`: the mean Menger curvature of the moved points (the setup cell's `menger`);
- `second_difference_length`: the mean length √((Δ²x)² + (Δ²y)²) of the circle's (unmoved) second differences;
- `kappa_ds_squared`: κ·Δs² for the circle, with κ = 1 / 50 and Δs the mean distance between neighbouring points;
- `relative_gap`: |`second_difference_length` − `kappa_ds_squared`| / `kappa_ds_squared`;
- `what_second_differences_measure`: one sentence: what does a second difference of a line's points measure, and why is it only a stand-in for curvature on a real track whose points are not evenly spaced?

**Prediction.** Going anticlockwise the left is the inside, so the left normal of the circle is −(cos θ, sin θ), pointing at the centre.
Moving every point 3 m inward gives a circle of radius 47 m, curvature 1/47 ≈ 0.02128 1/m.
On a regular 300-gon of radius R the chord is Δs = 2R sin(π/300) and the second difference has length 4R sin²(π/300), which is exactly (1/R)·Δs²: the stand-in is exact on an evenly spaced circle, so the relative gap should be rounding only.

In [ ]:
def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


theta = 2 * np.pi * np.arange(300) / 300
cx, cy = 50 * np.cos(theta), 50 * np.sin(theta)
cnx, cny = normals(cx, cy)
points_inward = max(np.max(np.abs(cnx + cx / 50)), np.max(np.abs(cny + cy / 50))) <= 1e-9
inner_x, inner_y = cx + 3.0 * cnx, cy + 3.0 * cny
inner_radius = float(np.mean(np.hypot(inner_x, inner_y)))
inner_curvature = float(np.mean(menger(inner_x, inner_y)))
second_difference_length = float(np.mean(np.hypot(second_difference(cx), second_difference(cy))))
mean_ds = float(np.mean(np.hypot(np.roll(cx, -1) - cx, np.roll(cy, -1) - cy)))
kappa_ds_squared = (1 / 50) * mean_ds ** 2
relative_gap = abs(second_difference_length - kappa_ds_squared) / kappa_ds_squared
what_second_differences_measure = (
    "A second difference measures how much the line bends between three neighbouring points (about "
    "kappa * ds^2, pointing into the turn), but it also grows and shrinks with the spacing ds, so where a "
    "track's points are not evenly spaced it mixes the spacing in with the curvature and is only a stand-in."
)

print(f"normals point at the centre: {points_inward}")
print(f"moved 3 m inward: mean radius {inner_radius:.6f} m, mean curvature {inner_curvature:.6f} 1/m "
      f"(1 / 47 = {1 / 47:.6f})")
print(f"second differences: mean length {second_difference_length:.6e} m; kappa ds^2 {kappa_ds_squared:.6e} m; "
      f"relative gap {relative_gap:.1e}")
print(what_second_differences_measure)

# independent checks: a regular 300-gon of radius 50 has chord 2 R sin(pi / 300), and its
# second differences have length 4 R sin^2(pi / 300), which is exactly kappa * chord^2
chord = 2 * 50 * math.sin(math.pi / 300)
assert abs(mean_ds - chord) < 1e-12
assert abs(second_difference_length - 4 * 50 * math.sin(math.pi / 300) ** 2) < 1e-12
assert points_inward
assert abs(inner_radius - 47.0) < 1e-9 and abs(inner_curvature - 1 / 47) < 1e-9
assert relative_gap < 1e-9

## Exercise 3 - The smoothness objective as a matrix

For a lap with points (x, y), left normals (N_x, N_y) and a basis B (n × m), the line with control values c has points x' = x + N_x (B c), y' = y + N_y (B c), and the smoothness objective Σ (Δ²x')² + (Δ²y')² equals ‖A c − b‖² with

- A = the rows of `second_difference(nx[:, None] * B)` stacked **above** the rows of `second_difference(ny[:, None] * B)` (2n × m), and
- b = −(the n entries of `second_difference(x)` followed by the n entries of `second_difference(y)`).

Write `smoothness_system(x, y, basis)`, returning `(A, b)`.

Use `test-circuit` (`track = load_track("test-circuit")`, its `track.x`, `track.y`) with `spline_basis(n, 10)`.
The reference's control values are a module constant of `agents/unit-09.py`; read them like this:

```python
with shim_installed():
    Unit09, module_name = load_submission(UNIT_09)
shipped = np.array(sys.modules[module_name].LINE_CONTROL)
```

Define:

- `smoothness_system`: the function;
- `A`, `b`: its result on `test-circuit`;
- `objective_centre`: ‖A c − b‖² for c = 0 (the centreline), and `direct_centre`: Σ (Δ²x)² + (Δ²y)² computed directly from the centreline's points;
- `objective_line`: ‖A c − b‖² for c = `shipped`, and `direct_line`: Σ (Δ²x')² + (Δ²y')² computed directly from the line's points;
- `largest_gap`: the larger of |`objective_centre` − `direct_centre`| and |`objective_line` − `direct_line`|;
- `objective_reduction`: 1 − `objective_line` / `objective_centre`;
- `curvature_reduction`: 1 − (the line's Σκ²Δs) / (the centreline's Σκ²Δs), from the setup cell's `line_summary`;
- `same_story`: one sentence: do the stand-in and the curvature agree on the direction of the change, and on its size?

**Prediction.** ‖A c − b‖² is the objective written as a matrix, so it should equal the direct sum to rounding for **any** c, the centreline and the line alike.
The lesson measured Σκ²Δs falling by 19% (0.2880 → 0.2326); the line was chosen to minimise the second differences, so they should fall in the same direction and probably by more, since that is the number the solver pushed down.

In [ ]:
def smoothness_system(x, y, basis):
    nx, ny = normals(x, y)
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    return A, b


track = load_track("test-circuit")
n = len(track)
x, y = track.x, track.y
basis = spline_basis(n, 10)
m = basis.shape[1]
A, b = smoothness_system(x, y, basis)
with shim_installed():
    Unit09, module_name = load_submission(UNIT_09)
shipped = np.array(sys.modules[module_name].LINE_CONTROL)

nx, ny = normals(x, y)
line_x, line_y = x + nx * (basis @ shipped), y + ny * (basis @ shipped)
objective_centre = float(np.sum((A @ np.zeros(m) - b) ** 2))
direct_centre = float(np.sum(second_difference(x) ** 2 + second_difference(y) ** 2))
objective_line = float(np.sum((A @ shipped - b) ** 2))
direct_line = float(np.sum(second_difference(line_x) ** 2 + second_difference(line_y) ** 2))
largest_gap = max(abs(objective_centre - direct_centre), abs(objective_line - direct_line))
objective_reduction = 1 - objective_line / objective_centre
curvature_reduction = 1 - line_summary(line_x, line_y)[0] / line_summary(x, y)[0]
same_story = (
    f"They agree on the direction (both fall) but not on the size: the second differences fall by "
    f"{objective_reduction:.0%} while the curvature the planner will see falls by only {curvature_reduction:.0%}, "
    "because the stand-in is the number the solver pushed down, and it also counts the spacing of the points."
)

print(f"A is {A.shape}, b has {b.size} entries, {m} control values")
print(f"centreline: objective {objective_centre:.6f}, direct {direct_centre:.6f} m^2")
print(f"the line:   objective {objective_line:.6f}, direct {direct_line:.6f} m^2")
print(f"largest gap {largest_gap:.1e}; objective -{objective_reduction:.1%}, sum kappa^2 ds -{curvature_reduction:.1%}")
print(same_story)

# independent checks: the matrix form is the objective for any c; the lesson's curvature numbers
assert A.shape == (2 * n, m) and b.shape == (2 * n,) and m == 133
assert abs(objective_centre - float(np.sum(b ** 2))) < 1e-12
assert largest_gap < 1e-9 * objective_centre
assert abs(line_summary(x, y)[0] - 0.2880) <= 5e-5 and abs(line_summary(line_x, line_y)[0] - 0.2326) <= 0.002
assert objective_reduction > curvature_reduction > 0.15

## Exercise 4 - QR, and why it gives the same answer

Unit 8 taught that with `Q, R = np.linalg.qr(A)` (Q is 2n × m with orthonormal columns, R is m × m upper triangular), ‖A c − b‖² = ‖R c − Qᵀb‖² + const for **every** c, the constant being ‖b‖² − ‖Qᵀb‖².
Check it on Exercise 3's `A` and `b`.

Define:

- `Q`, `R`: the factors;
- `q_shape`, `r_shape`: their shapes (tuples);
- `orthonormal_error`: the largest |G[i, k] − 1| for i = k and |G[i, k]| for i ≠ k, over the entries of G = QᵀQ (m × m; QᵀQ is the identity when the columns are orthonormal);
- `below_diagonal`: the largest |R[i, k]| over i > k (R's entries below its diagonal);
- `test_controls`: a list of three control vectors: all zeros, `shipped`, and `3.0 * np.sin(np.arange(m))` (any c will do);
- `differences`: a list of ‖A c − b‖² − ‖R c − Qᵀb‖² for the three, in that order;
- `outside_part`: ‖b‖² − ‖Qᵀb‖²;
- `largest_relative_mismatch`: the largest |difference − `outside_part`| / ‖b‖² over the three;
- `lstsq_gap`: the largest |entry| of the difference between the unbounded solutions `np.linalg.lstsq(A, b, rcond=None)[0]` and `np.linalg.lstsq(R, Q.T @ b, rcond=None)[0]`;
- `why_r_is_enough`: one or two sentences: the constant does not depend on c, so what does that say about the c that minimises either side, with or without bounds, and why is the R problem cheaper?

**Prediction.** `np.linalg.qr` gives the reduced factorisation: Q is 2662 × 133 and R is 133 × 133, QᵀQ is the identity to rounding and R has exact zeros below its diagonal.
The three differences should be the same number, ‖b‖² − ‖Qᵀb‖², to rounding, and the two unbounded solutions should agree to rounding too.

In [ ]:
Q, R = np.linalg.qr(A)
q_shape, r_shape = Q.shape, R.shape
G = Q.T @ Q
identity = np.zeros((m, m))
for k in range(m):
    identity[k, k] = 1.0
orthonormal_error = float(np.max(np.abs(G - identity)))
below_diagonal = max(abs(R[i, k]) for i in range(m) for k in range(i))
test_controls = [np.zeros(m), shipped, 3.0 * np.sin(np.arange(m))]
Qtb = Q.T @ b
differences = [float(np.sum((A @ c - b) ** 2) - np.sum((R @ c - Qtb) ** 2)) for c in test_controls]
outside_part = float(np.sum(b ** 2) - np.sum(Qtb ** 2))
largest_relative_mismatch = max(abs(d - outside_part) for d in differences) / float(np.sum(b ** 2))
lstsq_gap = float(np.max(np.abs(np.linalg.lstsq(A, b, rcond=None)[0] - np.linalg.lstsq(R, Qtb, rcond=None)[0])))
why_r_is_enough = (
    "The two objectives differ by the same constant for every c, so whatever c makes one smallest makes the "
    "other smallest too, with or without bounds; and the R problem has only m equations instead of 2n "
    f"({m} against {2 * n} here), so every step of the solver works on a matrix about {2 * n // m} times smaller."
)

print(f"Q {q_shape}, R {r_shape}; |Q^T Q - I| <= {orthonormal_error:.1e}; below R's diagonal <= {below_diagonal}")
print("differences:", [f"{d:.10f}" for d in differences], f"outside part {outside_part:.10f}")
print(f"largest relative mismatch {largest_relative_mismatch:.1e}; lstsq gap {lstsq_gap:.1e} m")
print(why_r_is_enough)

# independent checks: the identity holds for every c tried, and the two unbounded problems agree
assert q_shape == (2 * n, m) and r_shape == (m, m)
assert orthonormal_error < 1e-12 and below_diagonal < 1e-12
assert len(differences) == 3 and max(differences) - min(differences) < 1e-9
assert abs(outside_part - (float(np.sum(b ** 2)) - float(np.sum(Qtb ** 2)))) == 0.0 and outside_part > 0.0
assert largest_relative_mismatch < 1e-12 and lstsq_gap < 1e-9

## Exercise 5 - Solve a line on `test-circuit-seed1`

Write `solve_line(track, margin=2.5, pin=30)`, the lesson's solver, returning the control values as a NumPy array:

1. the basis `spline_basis(n, 10)`, the normals, and `smoothness_system(track.x, track.y, basis)`;
2. the bounds: control value k's support is the waypoints where column k of the basis is > 0; its bound h_k is the smallest `track.half_width[i] - margin` over its support, except that h_k = 0 when its support contains any of the `pin` waypoints at each end of the lap (indices 0 … pin − 1 and n − pin … n − 1);
3. `Q, R = np.linalg.qr(A)`, then `lsq_linear(R, Q.T @ b, bounds=(-h - 1e-12, h + 1e-12), method="bvls")`, and return its `.x` (the 1e-12 keeps every lower bound strictly below its upper bound, as `lsq_linear` requires).

Solve `test-circuit-seed1` (`seed1 = load_track("test-circuit-seed1")`).
Then solve `test-circuit` and compare with the reference agent's offsets: build an agent from the waypoints, `Unit09(track.waypoints, None)` (Exercise 3's class), whose `.offsets` is the list of its per-waypoint offsets.

Define:

- `solve_line`: the function;
- `seed1_control`, `seed1_offsets`: the control values and the offsets `spline_basis(len(seed1), 10) @ seed1_control`;
- `inside_margin`: `True` when every |offset| ≤ half-width − 2.5 + 1e-9;
- `pinned_start`: `True` when every |offset| on the 30 waypoints at each end of the lap is ≤ 1e-9;
- `on_bound_count`: the number of control values with h_k > 0 and |c_k| within 1e-9 of h_k;
- `centre_numbers`, `line_numbers`: `line_summary` of the centreline and of the line's points;
- `curvature_change`: `line_numbers[0] / centre_numbers[0] - 1`; `shorter_by`: the centreline's lap minus the line's (m); `tightest_change`: the line's tightest radius minus the centreline's (m);
- `reference_gap`: the largest |difference| between your `test-circuit` offsets and the reference agent's `.offsets` (m);
- `fig`, `ax`: the offsets of `test-circuit-seed1`'s line against `seed1.s` (the distance along the centreline), with ± (half-width − 2.5) drawn for comparison, axis labels and a title.

**Prediction.** `test-circuit-seed1` is a sibling of `test-circuit` (1313 waypoints, so 131 control values), so its line should look like the lesson's: the 9 control values whose supports touch the start pinned, about 40 more on a bound, Σκ²Δs down by roughly a fifth, a lap some 25 m shorter, and possibly a tightest corner that gets tighter, as on `test-circuit`.
On `test-circuit` the solver is the lesson's, so it should re-solve the reference's offsets to rounding.

In [ ]:
def control_bounds(basis, half_width, margin, pin):
    # h_k: the smallest half_width - margin over control value k's support, 0 if the support touches the pin.
    rows, columns = basis.shape
    high = np.zeros(columns)
    for k in range(columns):
        support = np.nonzero(basis[:, k] > 0)[0]
        if np.min(support) < pin or np.max(support) >= rows - pin:
            high[k] = 0.0
        else:
            high[k] = np.min(half_width[support] - margin)
    return high


def bounded_solve(track, basis, high):
    # QR, then bounded-variable least squares on R c = Q^T b.
    A, b = smoothness_system(track.x, track.y, basis)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


def solve_line(track, margin=2.5, pin=30):
    basis = spline_basis(len(track), 10)
    return bounded_solve(track, basis, control_bounds(basis, track.half_width, margin, pin))


def offset_points(track, offsets):
    # the line's points: every waypoint moved its offset along its left normal
    nx, ny = normals(track.x, track.y)
    return track.x + offsets * nx, track.y + offsets * ny


seed1 = load_track("test-circuit-seed1")
seed1_basis = spline_basis(len(seed1), 10)
seed1_control = solve_line(seed1)
seed1_offsets = seed1_basis @ seed1_control
seed1_high = control_bounds(seed1_basis, seed1.half_width, 2.5, 30)
inside_margin = float(np.max(np.abs(seed1_offsets) - (seed1.half_width - 2.5))) <= 1e-9
pinned_start = max(np.max(np.abs(seed1_offsets[:30])), np.max(np.abs(seed1_offsets[len(seed1) - 30:]))) <= 1e-9
on_bound_count = int(np.sum((seed1_high > 0) & (np.abs(np.abs(seed1_control) - seed1_high) <= 1e-9)))
centre_numbers = line_summary(seed1.x, seed1.y)
line_numbers = line_summary(*offset_points(seed1, seed1_offsets))
curvature_change = line_numbers[0] / centre_numbers[0] - 1
shorter_by = centre_numbers[1] - line_numbers[1]
tightest_change = line_numbers[2] - centre_numbers[2]

control = solve_line(track)
agent = Unit09(track.waypoints, None)
reference_gap = float(np.max(np.abs(basis @ control - np.array(agent.offsets))))

fig, ax = plt.subplots(figsize=(11, 3.5))
room = seed1.half_width - 2.5
ax.plot(seed1.s, room, color="grey", linestyle="--", label="+/- (half-width - 2.5 m)")
ax.plot(seed1.s, -room, color="grey", linestyle="--")
ax.plot(seed1.s, seed1_offsets, label="the line's offset")
ax.set_xlabel("distance along the centreline (m)")
ax.set_ylabel("offset n (m, left +)")
ax.set_title("test-circuit-seed1: the smoothest line inside a 2.5 m margin")
ax.legend()
plt.show()

print(f"{len(seed1)} waypoints, {len(seed1_control)} control values, {int(np.sum(seed1_high == 0))} pinned, "
      f"{on_bound_count} more on a bound")
print(f"inside the margin: {inside_margin}; pinned at the start: {pinned_start}")
for name, (total, length, radius) in (("centreline", centre_numbers), ("the line", line_numbers)):
    print(f"{name:10}: sum kappa^2 ds {total:.4f} 1/m, lap {length:.1f} m, tightest radius {radius:.1f} m")
print(f"curvature {curvature_change:+.1%}, {shorter_by:.1f} m shorter, tightest radius {tightest_change:+.1f} m")
print(f"test-circuit: the largest difference from the reference's offsets {reference_gap:.1e} m")


def objective(track, basis, c):
    A, b = smoothness_system(track.x, track.y, basis)
    return float(np.sum((A @ c - b) ** 2))


# independent checks: the bounds hold, the line beats feasible alternatives (the centreline and half the
# line, which are inside the same bounds), and it re-solves the reference
assert len(seed1_control) == len(seed1) // 10 == 131 and int(np.sum(seed1_high == 0)) == 9
assert np.max(np.abs(seed1_control) - seed1_high) <= 1e-11  # the solver's 1e-12 slack
assert inside_margin and pinned_start and on_bound_count == 42
best = objective(seed1, seed1_basis, seed1_control)
assert best < objective(seed1, seed1_basis, 0.5 * seed1_control) < objective(seed1, seed1_basis, 0 * seed1_control)
assert -0.20 < curvature_change < -0.15 and 20.0 < shorter_by < 30.0 and tightest_change < 0.0
assert reference_gap <= 1e-6

## Exercise 6 - Following a line, and the clean rule

**The `REACHED_RADIUS` trap.**
Use `test-circuit` and the reference's offsets (`np.array(agent.offsets)` for the agent of Exercise 5, or `spline_basis(n, 10) @ shipped`).
Count the line points that have **no** centreline waypoint within `REACHED_RADIUS` (the smallest distance from the line point to any of the n centreline waypoints is ≥ 3.0 m).

**The clean rule.**
racekit's telemetry `offset` is the car's signed distance from the **centreline** (left positive); a row with progress p is at waypoint (p + 1) mod n.
Write:

- `line_error(result, offsets)`: the largest |`offset` − offsets[(p + 1) mod n]| over the driving rows (`held == 0`), n = `len(offsets)`;
- `clean(result, offsets)`: finished, 0 collisions, and `line_error` at most `CLEAN_ERROR`.

Race the reference with `repeat_races(UNIT_09, "test-circuit", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True), max_seconds=450.0)`.

Define:

- `line_error`, `clean`: the functions;
- `unreachable`: the count above, and `unreachable_fraction`: it divided by n;
- `reference_results`: the `RepeatResult` (Exercise 9 reuses it);
- `errors_from_line`: the three races' `line_error` with the reference's offsets (m), in seed order;
- `errors_from_centre`: the three races' `line_error` with offsets of all zeros (the max |offset from the centreline|);
- `unit_7_verdicts`: for each race, `True` when it finished with 0 collisions and its max |offset from the centreline| was at most 2 m (Unit 7's rule);
- `line_verdicts`: for each race, `clean(result, reference offsets)`;
- `explain`: two sentences: why would an agent that advanced its index on the **centreline's** waypoints get stuck on this line, and why is Unit 7's rule the wrong rule for a line race?

**Prediction.** The lesson counted a third of the lap where the line is too far from the centreline for a centreline waypoint to be within 3 m.
The reference's seed-0 race was 1.52 m from its own line at worst and 279.60 s, so three seeds should be about 1.5 m from the line, but more than 3.5 m from the centreline (the line itself goes 3.5 m out): Unit 7's rule says "not clean" for all three, the line rule "clean".

In [ ]:
def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


reference_offsets = np.array(agent.offsets)
ref_x, ref_y = offset_points(track, reference_offsets)
unreachable = 0
for i in range(n):
    if np.min(np.hypot(x - ref_x[i], y - ref_y[i])) >= REACHED_RADIUS:
        unreachable += 1
unreachable_fraction = unreachable / n
reference_results = repeat_races(UNIT_09, "test-circuit", n=3, seeds=(0, 1, 2),
                                 car_factory=lambda: carla_like(noise=True), max_seconds=450.0)
centreline = np.zeros(n)
errors_from_line = [line_error(result, reference_offsets) for result in reference_results.results]
errors_from_centre = [line_error(result, centreline) for result in reference_results.results]
unit_7_verdicts = [result.finished and result.collisions == 0 and line_error(result, centreline) <= 2.0
                   for result in reference_results.results]
line_verdicts = [clean(result, reference_offsets) for result in reference_results.results]
explain = (
    f"Where the line has no centreline waypoint within 3 m ({unreachable} of {n} points, a third of the lap), "
    "a car on the line never comes within REACHED_RADIUS of the centreline's next waypoint, so an index "
    "advanced on the centreline's waypoints stops advancing and the agent keeps steering for a point it has "
    "already passed. "
    "Unit 7's rule measures the distance from the centreline, which in a line race is the line's own planned "
    f"offset (up to {np.max(np.abs(reference_offsets)):.1f} m) plus the follower's error, so it calls every race "
    f"\"not clean\" ({max(errors_from_centre):.2f} m) while the error that matters, the distance from the car's own "
    f"line, is at most {max(errors_from_line):.2f} m."
)

print(f"{unreachable} of {n} line points ({unreachable_fraction:.1%}) have no centreline waypoint within 3 m")
print(reference_results, [round(float(t), 2) for t in reference_results.times], "s")
print("from the line:", [round(e, 3) for e in errors_from_line])
print("from the centreline:", [round(e, 3) for e in errors_from_centre])
print("Unit 7's rule:", unit_7_verdicts, "; the line rule:", line_verdicts)
print(explain)

# independent checks: the lesson's count and seed-0 race, and the two rules' verdicts
assert unreachable == 441 and 0.3 < unreachable_fraction < 0.35  # the lesson: more than 300, a third of the lap
assert abs(reference_results.times[0] - 279.60) <= TICK  # the lesson's seed-0 race
assert abs(reference_results.median - 279.60) <= TICK and reference_results.spread <= 2 * TICK
assert max(errors_from_line) <= CLEAN_ERROR and min(errors_from_centre) > 3.5
assert unit_7_verdicts == [False, False, False] and line_verdicts == [True, True, True]

## Exercise 7 - The honest comparison on `test-circuit-seed1`

Is the line's gain at the same f a `test-circuit` accident?
Race both agents on `test-circuit-seed1` at f = 0.54, with that track's measured grip:

- the line: `variant(UNIT_09, "seed1-line", LINE_CONTROL=tuple(float(v) for v in seed1_control), MU_G=MEASURED_MU_G["test-circuit-seed1"], MU_FRACTION=0.54)`;
- the centreline: `variant(UNIT_08, "seed1-centre", MU_G=MEASURED_MU_G["test-circuit-seed1"], MU_FRACTION=0.54)`;

each with `repeat_races` on `"test-circuit-seed1"`, seeds 0, 1 and 2 (`n=3, seeds=(0, 1, 2)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=450.0`.

Define:

- `line_results`, `centre_results`: the two `RepeatResult` objects;
- `all_clean`: `True` when all six races are clean by Exercise 6's `clean` (the line's races against `seed1_offsets`, the centreline's against zeros);
- `medians`, `spreads`: dicts `{"line": ..., "centre": ...}` of the median race time and the spread (s);
- `line_better`: `noise_aware_better(line_results.times, centre_results.times)`;
- `gain_s`: the centreline's median minus the line's (s);
- `compared_with_test_circuit`: one sentence: the lesson measured about 25 s on `test-circuit` at the same f; is this track's gain similar, and why might the gain differ from track to track?

**Prediction.** The lesson's table gives the line at 0.54 on `test-circuit-seed1` 276.20 s (median over seeds 0–4).
On `test-circuit` the centreline at the same f was about 25 s slower; `test-circuit-seed1` is a sibling track with a similar line (Exercise 5: Σκ²Δs −18%, 24 m shorter), so I expect a similar gain, a little under 25 s, far beyond spreads of a tick or a few.

In [ ]:
line_results = repeat_races(
    variant(UNIT_09, "seed1-line", LINE_CONTROL=tuple(float(v) for v in seed1_control),
            MU_G=MEASURED_MU_G["test-circuit-seed1"], MU_FRACTION=0.54),
    "test-circuit-seed1", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True), max_seconds=450.0)
centre_results = repeat_races(
    variant(UNIT_08, "seed1-centre", MU_G=MEASURED_MU_G["test-circuit-seed1"], MU_FRACTION=0.54),
    "test-circuit-seed1", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True), max_seconds=450.0)
seed1_centreline = np.zeros(len(seed1))
all_clean = False not in ([clean(result, seed1_offsets) for result in line_results.results]
                          + [clean(result, seed1_centreline) for result in centre_results.results])
line_times = [result.total_time for result in line_results.results]
centre_times = [result.total_time for result in centre_results.results]
medians = {"line": float(np.median(line_times)), "centre": float(np.median(centre_times))}
spreads = {"line": max(line_times) - min(line_times), "centre": max(centre_times) - min(centre_times)}
line_better = noise_aware_better(line_results.times, centre_results.times)
gain_s = medians["centre"] - medians["line"]
compared_with_test_circuit = (
    f"Yes: {gain_s:.1f} s here against about 25 s on test-circuit, both about 8% of the centreline's time, but the "
    "gain need not be the same everywhere, because the line straightens only the corners the road is wide enough "
    "to straighten, and each track has its own corners, widths and measured grip."
)

print("line:", [round(float(t), 2) for t in line_times], "s; centreline:",
      [round(float(t), 2) for t in centre_times], "s")
print("from their lines:", [round(line_error(result, seed1_offsets), 2) for result in line_results.results],
      [round(line_error(result, seed1_centreline), 2) for result in centre_results.results], "m")
print(f"all clean: {all_clean}; medians: line {medians['line']:.2f} s, centreline {medians['centre']:.2f} s; "
      f"spreads: line {spreads['line']:.2f} s, centreline {spreads['centre']:.2f} s")
print(f"line better beyond the noise: {line_better}; gain {gain_s:.2f} s")
print(compared_with_test_circuit)

# independent checks: the RepeatResults' own medians and spreads, and the lesson's table (276.20 s over seeds 0-4)
assert all_clean and line_better
assert medians["line"] == line_results.median and medians["centre"] == centre_results.median
assert abs(spreads["line"] - line_results.spread) < 1e-9 and abs(spreads["centre"] - centre_results.spread) < 1e-9
assert abs(medians["line"] - 276.20) <= 2 * TICK and abs(medians["centre"] - 300.25) <= TICK
assert 20.0 < gain_s < 30.0 and gain_s > 50 * max(spreads.values())

## Exercise 8 - Challenge: a late-apex window

The lesson's local widening opened a **symmetric** window, 60 waypoints either side of every point tighter than 60 m radius.
Racing drivers prefer a **late apex**: turn in later, and use the road on the way **out** of the corner, where the car is accelerating.
Try a window with more waypoints after the tight point than before, on `monza-like` (where the symmetric window races clean).

Write:

- `window_mask(x, y, before, after, radius=60.0)`: a boolean NumPy array, `True` at every waypoint k with i − before ≤ k ≤ i + after (mod n) for some waypoint i whose |Menger curvature| is > 1 / radius ("after" is the direction of travel);
- `local_line(track, near, margin=2.5, pin=30)`: Exercise 5's solver with one more rule: h_k = 0 also when control value k's support contains any waypoint where `near` is `False`.

Solve two lines on `monza-like`: symmetric, `window_mask(..., 60, 60)`, and late, `window_mask(..., 30, 90)`.
Race each as `variant(UNIT_09, ..., LINE_CONTROL=..., MU_G=MEASURED_MU_G["monza-like"], MU_FRACTION=0.60)` with `repeat_races` on `"monza-like"`, seeds 0 and 1 (`n=2, seeds=(0, 1)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=900.0`.

Define:

- `window_mask`, `local_line`: the functions;
- `symmetric_control`, `late_control`: the two lines' control values;
- `summaries`: a dict `{"symmetric": ..., "late": ...}` of each line's `line_summary` on its points;
- `symmetric_results`, `late_results`: the two `RepeatResult` objects;
- `windows_clean`: `True` when all four races are clean by Exercise 6's `clean`, each against its own line's offsets;
- `window_medians`: a dict `{"symmetric": ..., "late": ...}` of the median race times (s);
- `late_better`: `noise_aware_better(late_results.times, symmetric_results.times)`;
- `why_late`: two sentences: which line is smoother by Σκ²Δs, which is faster, and what does that say about using the smoothness objective alone to choose a line?

**Prediction.** Both windows are 121 waypoints wide, so they free the same amount of road around every tight point, just placed differently.
The smoothness objective does not know which way the car drives, so it has no reason to prefer the late window; I expect the symmetric line to be about as smooth or smoother.
The race is what knows the direction: a late apex keeps the line in the middle on the way in, where the car brakes, and uses the road on the way out, where it accelerates, so I expect the late line to be a little faster, by a second or two over a 490 s race.

In [ ]:
def window_mask(x, y, before, after, radius=60.0):
    count = len(x)
    near = np.zeros(count, dtype=bool)
    for i in np.nonzero(np.abs(menger(x, y)) > 1.0 / radius)[0]:
        near[np.arange(i - before, i + after + 1) % count] = True
    return near


def local_line(track, near, margin=2.5, pin=30):
    basis = spline_basis(len(track), 10)
    high = control_bounds(basis, track.half_width, margin, pin)
    for k in range(basis.shape[1]):
        if np.sum(~near[basis[:, k] > 0]) > 0:
            high[k] = 0.0
    return bounded_solve(track, basis, high)


monza = load_track("monza-like")
monza_basis = spline_basis(len(monza), 10)
symmetric_control = local_line(monza, window_mask(monza.x, monza.y, 60, 60))
late_control = local_line(monza, window_mask(monza.x, monza.y, 30, 90))
window_offsets = {"symmetric": monza_basis @ symmetric_control, "late": monza_basis @ late_control}
summaries = {name: line_summary(*offset_points(monza, offsets)) for name, offsets in window_offsets.items()}
symmetric_results = repeat_races(
    variant(UNIT_09, "monza-symmetric", LINE_CONTROL=tuple(float(v) for v in symmetric_control),
            MU_G=MEASURED_MU_G["monza-like"], MU_FRACTION=0.60),
    "monza-like", n=2, seeds=(0, 1), car_factory=lambda: carla_like(noise=True), max_seconds=900.0)
late_results = repeat_races(
    variant(UNIT_09, "monza-late", LINE_CONTROL=tuple(float(v) for v in late_control),
            MU_G=MEASURED_MU_G["monza-like"], MU_FRACTION=0.60),
    "monza-like", n=2, seeds=(0, 1), car_factory=lambda: carla_like(noise=True), max_seconds=900.0)
windows_clean = False not in ([clean(result, window_offsets["symmetric"]) for result in symmetric_results.results]
                              + [clean(result, window_offsets["late"]) for result in late_results.results])
window_medians = {"symmetric": symmetric_results.median, "late": late_results.median}
late_better = noise_aware_better(late_results.times, symmetric_results.times)
why_late = (
    f"The symmetric line is the smoother by sum kappa^2 ds ({summaries['symmetric'][0]:.4f} against "
    f"{summaries['late'][0]:.4f} 1/m), but the late-apex line is the faster ({window_medians['late']:.2f} against "
    f"{window_medians['symmetric']:.2f} s, beyond the noise). "
    "The smoothness objective does not know which way the car drives or where it brakes and accelerates, so it is a "
    "stand-in for lap time, not lap time: it can build a line, but only a race can choose between lines."
)

for name, results in (("symmetric", symmetric_results), ("late", late_results)):
    total, length, radius = summaries[name]
    errors = [round(line_error(result, window_offsets[name]), 2) for result in results.results]
    print(f"{name:9}: sum kappa^2 ds {total:.4f} 1/m, lap {length:.1f} m, tightest radius {radius:.1f} m; "
          f"races {[round(float(t), 2) for t in results.times]} s, {errors} m from the line")
print(f"all clean: {windows_clean}; medians: symmetric {window_medians['symmetric']:.3f} s, "
      f"late {window_medians['late']:.2f} s; late better beyond the noise: {late_better}")
print(why_late)

# independent checks: the windows are the same width, the late one is shifted forward; the lesson's 487.55 s
symmetric_near = window_mask(monza.x, monza.y, 60, 60)
late_near = window_mask(monza.x, monza.y, 30, 90)
first_tight = int(np.nonzero(np.abs(menger(monza.x, monza.y)) > 1.0 / 60.0)[0][0])
assert symmetric_near[(first_tight - 60) % len(monza)] and not late_near[(first_tight - 60) % len(monza)]
assert np.max(np.abs(window_offsets["late"][~late_near])) <= 1e-9
assert np.max(np.abs(window_offsets["symmetric"][~symmetric_near])) <= 1e-9
assert np.max(np.abs(window_offsets["late"]) - (monza.half_width - 2.5)) <= 1e-9
assert windows_clean and late_better
assert abs(window_medians["symmetric"] - 487.55) <= TICK
assert summaries["symmetric"][0] < summaries["late"][0] and window_medians["late"] < window_medians["symmetric"]

## Exercise 9 - Challenge: a 2.0 m margin

The line keeps 2.5 m from each edge.
A narrower margin gives the line more road, so it can be straighter: is it worth the risk?

Solve `test-circuit`'s line with `solve_line(track, margin=2.0)` and race it with the reference's grip and f, `variant(UNIT_09, "margin-2.0", LINE_CONTROL=tuple(float(v) for v in narrow_control))`, with `repeat_races` on `"test-circuit"`, seeds 0, 1 and 2, `car_factory=lambda: carla_like(noise=True)` and `max_seconds=450.0`.
Compare with Exercise 6's `reference_results` (the 2.5 m margin, the same seeds).

The car's **room to the edge** on a driving row is the half-width at its waypoint minus its distance from the centreline: `track.half_width[(p + 1) % n] - abs(offset)`.

Define:

- `narrow_control`, `narrow_offsets`: the 2.0 m line's control values and offsets;
- `narrow_numbers`: its `line_summary`;
- `narrow_results`: the `RepeatResult`;
- `narrow_clean`: `True` when its three races are clean by Exercise 6's `clean`;
- `margin_medians`: a dict from margin (2.5 and 2.0) to the median race time (s);
- `time_gained`: `margin_medians[2.5] - margin_medians[2.0]` (s);
- `beats_the_noise`: `noise_aware_better(narrow_results.times, reference_results.times)`;
- `closest_to_edge`: a dict from margin to the smallest room to the edge over the driving rows of its three races (m);
- `worth_it`: two or three sentences: is the narrower margin worth its risk? Weigh the time it gains against the room it gives up, and say what CARLA, whose car and track are not racekit's, would add to the risk.

**Prediction.** Half a metre more room on each side lets the solver straighten every corner a little more: a lower Σκ²Δs and a shorter lap, so a faster race, but only a little (the line already uses 3–3.5 m of room; 0.5 m more is about 15%), perhaps a second or two.
The price is the same half metre taken off the room to the edge: the follower's error (about 1.5 m) does not shrink, so the car should come about 0.5 m closer to the edge.

In [ ]:
narrow_control = solve_line(track, margin=2.0)
narrow_offsets = basis @ narrow_control
narrow_numbers = line_summary(*offset_points(track, narrow_offsets))
narrow_results = repeat_races(variant(UNIT_09, "margin-2.0", LINE_CONTROL=tuple(float(v) for v in narrow_control)),
                              "test-circuit", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True),
                              max_seconds=450.0)
narrow_clean = False not in [clean(result, narrow_offsets) for result in narrow_results.results]
margin_medians = {2.5: reference_results.median, 2.0: narrow_results.median}
time_gained = margin_medians[2.5] - margin_medians[2.0]
beats_the_noise = noise_aware_better(narrow_results.times, reference_results.times)


def room_to_edge(results):
    # the smallest half-width minus |offset| over the driving rows of every race
    rooms = []
    for result in results.results:
        telemetry = result.telemetry
        driving = telemetry["held"] == 0
        rooms.append(float(np.min(track.half_width[(telemetry["progress"][driving] + 1) % n]
                                  - np.abs(telemetry["offset"][driving]))))
    return min(rooms)


closest_to_edge = {2.5: room_to_edge(reference_results), 2.0: room_to_edge(narrow_results)}
worth_it = (
    f"In racekit the 2.0 m line gains {time_gained:.2f} s ({time_gained / margin_medians[2.5]:.1%} of the race), clean "
    f"and beyond the noise, but the car comes within {closest_to_edge[2.0]:.2f} m of the edge instead of "
    f"{closest_to_edge[2.5]:.2f} m: it gives up {closest_to_edge[2.5] - closest_to_edge[2.0]:.2f} m of the room that "
    "absorbs the follower's errors for well under 1% of the race. "
    "In CARLA the car, its tyres and the track's edges are not racekit's, so the follower's error there is not yet "
    "measured and could be larger; I would keep 2.5 m until a CARLA run shows how close its car really comes."
)

for margin, numbers in ((2.5, line_summary(ref_x, ref_y)), (2.0, narrow_numbers)):
    print(f"margin {margin} m: sum kappa^2 ds {numbers[0]:.4f} 1/m, lap {numbers[1]:.1f} m, "
          f"tightest radius {numbers[2]:.1f} m")
print(f"2.0 m races {[round(float(t), 2) for t in narrow_results.times]} s, "
      f"{[round(line_error(result, narrow_offsets), 2) for result in narrow_results.results]} m from the line")
print(f"clean: {narrow_clean}; medians: 2.5 m {margin_medians[2.5]:.2f} s, 2.0 m {margin_medians[2.0]:.2f} s; "
      f"gained {time_gained:.2f} s; beats the noise: {beats_the_noise}")
print(f"closest to the edge: 2.5 m margin {closest_to_edge[2.5]:.2f} m, 2.0 m margin {closest_to_edge[2.0]:.2f} m")
print(worth_it)

# independent checks: the narrower line is inside its own margin, smoother and shorter, faster, and closer
assert np.max(np.abs(narrow_offsets) - (track.half_width - 2.0)) <= 1e-9
assert np.max(np.abs(narrow_offsets)) > np.max(np.abs(reference_offsets))
assert narrow_numbers[0] < line_summary(ref_x, ref_y)[0] and narrow_numbers[1] < line_summary(ref_x, ref_y)[1]
assert narrow_clean and beats_the_noise and abs(margin_medians[2.0] - 277.90) <= TICK
assert 1.0 < time_gained < 3.0
assert 0.0 < closest_to_edge[2.0] < closest_to_edge[2.5] - 0.3